# Under the hood · What a regression coefficient actually tells you

**Reference notebook.** Not a meeting — this is where Meeting 1 sends you if you
pushed back on *"this course mostly declines explanation"* and want the argument in
full. Regression **does** feel explanatory, and that feeling deserves a real answer
rather than an assertion.

The short version: **a fitted coefficient does not mean what it sounds like it
means.** Not because regression is broken, but because a fitted equation is a
*sentence*, and sentences sound like explanations whether or not they are one.

Everything below runs on the **120 Ames houses from Meeting 1**, so you can check
every number yourself. It runs in Colab or locally, unchanged.

The route:

1. Ask the model a simple question and get two contradictory answers
2. Rule out the obvious escape hatch — that it is just noise
3. Find out where the contradiction actually comes from
4. Pin down what the surviving number *is* measuring
5. Try to see it directly in the houses, fail, then succeed
6. Say why none of it is an effect

You need no more than Meeting 1 to follow it.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

URL = ("https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026/"
       "main/course/data/ames_day1.csv")
ames = pd.read_csv(URL)

y     = ames["SalePrice"]        # what the house sold for
rooms = ames["TotRms_AbvGrd"]    # total rooms above ground
area  = ames["Gr_Liv_Area"]      # above-ground living area, sq ft

print(f"{len(ames)} houses")

## 1. A simple question with two answers

**What is a room worth?** It is the kind of question people expect regression to
answer, so let us just ask it.

In [ ]:
m1 = sm.OLS(y, sm.add_constant(rooms)).fit()
print(f"  price ~ rooms                 {m1.params['TotRms_AbvGrd']:>12,.0f}  per room")

**+$28,601 a room.** That is a sentence you could say out loud to a homeowner.

Now add one more predictor — square footage. Nothing about the houses changed; we
are looking at the same 120 sales.

In [ ]:
m2 = sm.OLS(y, sm.add_constant(ames[["TotRms_AbvGrd", "Gr_Liv_Area"]])).fit()
print(f"  price ~ rooms                 {m1.params['TotRms_AbvGrd']:>12,.0f}  per room")
print(f"  price ~ rooms + square feet   {m2.params['TotRms_AbvGrd']:>12,.0f}  per room")

**−$18,159 a room.** The sign flipped.

Same data. Same variable. One is worth $28,601 and one costs you $18,159, depending
on what else is in the model. **Both regressions are correct.** Neither is lying.

So "what is a room worth" was not a well-posed question, and we did not notice,
because the output looked like an answer.

## 2. The obvious objection first: is this just noise?

Rooms and square footage are obviously related — bigger houses have more rooms. When
two predictors carry overlapping information, the fit has trouble telling them apart,
and the estimates get imprecise. Maybe the negative number is just wobble.

**This is worth taking seriously before building anything on top of it.** Two checks.

### Check one: how much precision did we lose?

The standard tool is the **variance inflation factor**. The idea is simpler than the
name: regress one predictor on the others, see how well they predict it, and ask how
much that overlap inflates the variance of its coefficient.

$$\mathrm{VIF}_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is from regressing predictor $j$ on the rest. With only two predictors
that $R^2$ is just their squared correlation, so you can almost do it in your head.

- VIF = 1 → no overlap, no penalty
- VIF = 4 → the standard error is $\sqrt{4} = 2\times$ what it would have been

It is a **precision** statement, not a bias statement. Collinearity makes coefficients
harder to pin down; it does not make them wrong.

In [ ]:
r = rooms.corr(area)
vif_by_hand = 1 / (1 - r**2)

from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
X = sm.add_constant(ames[["TotRms_AbvGrd", "Gr_Liv_Area"]])

print(f"  corr(rooms, area)          {r:>8.3f}")
print(f"  R-squared between them     {r**2:>8.3f}")
print(f"  VIF by hand, 1/(1-R2)      {vif_by_hand:>8.2f}")
print(f"  VIF from statsmodels       {vif(X.values, 1):>8.2f}   <- same thing")
print(f"\n  so the standard error is {np.sqrt(vif_by_hand):.2f}x wider than it would be")
print("  if rooms and square footage were unrelated. Real, but survivable.")

### Check two: would the sign survive a different sample?

VIF says we lost precision. It does not say whether *this* estimate is trustworthy.
For that, resample the 120 houses with replacement, refit, and watch. If the negative
is an accident of these particular houses, the sign should wander.

In [ ]:
rng = np.random.default_rng(0)
boot = np.array([
    sm.OLS(s["SalePrice"], sm.add_constant(s[["TotRms_AbvGrd", "Gr_Liv_Area"]]))
      .fit().params["TotRms_AbvGrd"]
    for s in (ames.sample(len(ames), replace=True,
                          random_state=int(rng.integers(1e9))) for _ in range(2000))
])
b = m2.params["TotRms_AbvGrd"]
se = m2.bse["TotRms_AbvGrd"]
lo, hi = m2.conf_int().loc["TotRms_AbvGrd"]

print(f"  estimate {b:>12,.0f}   se {se:>9,.0f}   t {b/se:>6.2f}")
print(f"  95% CI  [{lo:,.0f}, {hi:,.0f}]  — does not contain zero")
print(f"  negative in {(boot < 0).mean():.1%} of 2000 resamples")

It barely wanders. **The negative is a real feature of these houses**, not noise, so
we cannot dismiss the contradiction. We have to explain it.

## 3. Where did the flip come from?

Here is where that correlation earns its keep. It is **not** a warning about
collinearity — we just checked that and it was mild. It is the *channel*.

If rooms and square footage were unrelated, adding square footage would leave the
rooms coefficient exactly where it was. Nothing would flip. The two answers differ
**only because the two predictors overlap**, and there is an exact accounting for how
much travels down that channel:

$$\underbrace{\beta^{\text{alone}}_{\text{rooms}}}_{+28{,}601}
  \;=\;
  \underbrace{\beta^{\text{with area}}_{\text{rooms}}}_{-18{,}159}
  \;+\;
  \underbrace{\beta_{\text{area}}}_{\$\text{ per sq ft}} \times
  \underbrace{\delta}_{\text{sq ft that come with a room}}$$

The last term is the part rooms was *standing in for*.

In [ ]:
delta = sm.OLS(area, sm.add_constant(rooms)).fit().params["TotRms_AbvGrd"]
b_area = m2.params["Gr_Liv_Area"]

print(f"  an extra room comes with           {delta:>10,.0f} sq ft")
print(f"  square footage is worth            {b_area:>10,.0f} $/sq ft")
print(f"  so rooms were standing in for      {b_area*delta:>10,.0f} $ of house")
print()
print(f"  {m2.params['TotRms_AbvGrd']:>10,.0f}   the room itself")
print(f"+ {b_area*delta:>10,.0f}   the square footage it drags along")
print(f"= {m2.params['TotRms_AbvGrd'] + b_area*delta:>10,.0f}   what 'rooms alone' reported")
print(f"\n  matches the +{m1.params['TotRms_AbvGrd']:,.0f} exactly: "
      f"{np.isclose(m1.params['TotRms_AbvGrd'], m2.params['TotRms_AbvGrd'] + b_area*delta)}")

**That is the whole mystery, dissolved.** The +$28,601 was never "the value of a
room." It was the value of *a room plus the 255 square feet that come with it*. Once
you pay for the square footage separately, the room on its own is a liability.

## 4. So what is the −$18,159 measuring?

Everyone says the coefficient is the effect of rooms "holding square footage
constant." That phrase gets repeated so often it stops being examined. It has a
precise meaning, and you can compute it directly.

**Take square footage out of both variables, then look at what is left.**

- Regress rooms on area, keep the residuals → *how many more rooms this house has
  than its size would predict*
- Regress price on area, keep the residuals → *how much more it sold for than its
  size would predict*
- Regress the second on the first

If "holding constant" means what people say it means, that should reproduce the
multiple-regression coefficient. It does — exactly. (This is the Frisch–Waugh–Lovell
result; the name matters less than the fact that you can watch it happen.)

In [ ]:
rooms_net = sm.OLS(rooms, sm.add_constant(area)).fit().resid
price_net = sm.OLS(y,     sm.add_constant(area)).fit().resid
fwl = sm.OLS(price_net, sm.add_constant(rooms_net)).fit().params.iloc[1]

print(f"  multiple regression        {m2.params['TotRms_AbvGrd']:>12,.2f}")
print(f"  regression on residuals    {fwl:>12,.2f}")
print(f"  identical                  {np.isclose(m2.params['TotRms_AbvGrd'], fwl)}")

So the coefficient is answering this question, and only this one:

> Among houses of a **given size**, do the ones carved into *more rooms than usual*
> sell for more or less?

Less. Those are houses with smaller rooms. That is a real, interpretable pattern
about Ames — and it is nothing like "what a room is worth."

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.axhline(0, color="0.85", lw=1); ax.axvline(0, color="0.85", lw=1)
ax.scatter(rooms_net, price_net, s=28, alpha=.75)
xs = np.linspace(rooms_net.min(), rooms_net.max(), 50)
ax.plot(xs, fwl * xs, lw=2, color="crimson")
ax.set_xlabel("more rooms than this house's size predicts")
ax.set_ylabel("more $ than this house's size predicts")
ax.set_title(f"What the coefficient sees · slope = {fwl:,.0f}")
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 5. Can you just look at the houses instead?

Reasonable instinct, and the one a sharp student will follow: forget the algebra,
group houses of similar size and compare. Let us try it the obvious way — three
size bands.

In [ ]:
band = pd.qcut(area, 3, labels=["small", "mid", "large"])
for lbl, g in ames.groupby(band, observed=True):
    med  = g["TotRms_AbvGrd"].median()
    diff = (g[g["TotRms_AbvGrd"] > med]["SalePrice"].mean()
            - g[g["TotRms_AbvGrd"] <= med]["SalePrice"].mean())
    print(f"  {lbl:<6} {g['Gr_Liv_Area'].min():>5.0f}-{g['Gr_Liv_Area'].max():<5.0f} sq ft"
          f"   more rooms − fewer rooms = {diff:>+11,.0f}")

**It does not work.** The signs are inconsistent and two of the three are positive —
the opposite of what the regression said.

The regression is not wrong, and neither is the arithmetic. The bins are. A band
1,600 square feet wide does **not** hold size constant: inside it, the houses with
more rooms are still the bigger ones, so we never removed the thing we meant to
remove.

Tighten the matching until "same size" really means same size — pairs within 40
square feet of each other:

In [ ]:
d = ames[["SalePrice", "TotRms_AbvGrd", "Gr_Liv_Area"]].to_numpy()
per_room = np.array([
    (d[i,0]-d[j,0]) / (d[i,1]-d[j,1])
    for i in range(len(d)) for j in range(i+1, len(d))
    if abs(d[i,2]-d[j,2]) <= 40 and d[i,1] != d[j,1]
])
print(f"  {len(per_room)} pairs within ±40 sq ft that differ in room count")
print(f"  mean   {per_room.mean():>+11,.0f} per extra room")
print(f"  median {np.median(per_room):>+11,.0f}")
print(f"  more rooms sold for less in {(per_room < 0).mean():.0%} of pairs")

Same direction as the regression. **The idea was right; the first attempt was too
blunt** — and that failure is worth more than the success. "Holding constant" is
extremely demanding, and eyeballing groups almost never achieves it. The regression
does it exactly, which is its real advantage, and also why it is so easy to believe
it has done something deeper than it has.

## 6. Why none of this is an effect

Everything above is about **comparisons between houses that already exist**. Line up
the ones of equal size and count their rooms.

An effect is a different object: the *same* house under two versions of the world.
In notation, the difference between

$$E[Y \mid \text{rooms} = r]  \qquad\text{and}\qquad E[Y \mid do(\text{rooms} = r)]$$

— observing a house with $r$ rooms, versus *making* a house have $r$ rooms. Those
agree only under assumptions that least squares neither requires nor checks.

And here the gap is not abstract. We measured it in §3: **an extra room comes with
about 255 more square feet.** So when the coefficient holds square footage fixed, it
is holding fixed precisely the thing that building a room would change. It answers a
question about houses that differ in rooms *without* differing in size — which is not
what anyone means when they ask what a room is worth.

### The line

> *A regression coefficient is not explanatory either. On observational data it is a
> conditional association. It only looks explanatory because you can read it aloud.*

A fitted equation is a **sentence**, and sentences have the grammar of causal claims.
"Each room is worth −$18,159" *sounds* like an explanation. A random forest produces
no sentence, so nobody is tempted to make one. **The linear model's interpretability
is syntactic, not epistemic** — it is a property of the notation, not of what you
learned.

### What this is not saying

It is **not** saying regression is useless, or that you can never learn why something
happens. Causal inference is a real field with real tools, and none of them are least
squares applied to whatever columns you had.

It *is* saying that reading a coefficient aloud is not the same as knowing why — and
that the difference is invisible from the output, which is what makes it dangerous.

### Where this comes back

**Week 10** takes interpretation seriously — permutation importance, partial
dependence, ICE, SHAP — and then spends the next meeting breaking every one of those
tools on correlated features and extrapolation. Everything above is the linear-model
version of that lesson, and it is the easy case: at least here you can write the model
down in closed form and still be misled.

If you want the honest habit in one sentence: **whenever you are about to say what a
coefficient means, say what two groups of houses you are comparing.** If you cannot,
you do not yet know what the number is.